# pr-work #4
## First name - Bohdan | Second name - Moshnenko | Group - CS-24

In [1]:
import pandas as pd
import numpy as np


# downloading csv
df = pd.read_csv(r"C:\Users\mbv16\Downloads\financial_regression.csv")
print(df.shape, df.dtypes)

(3904, 47) date                   object
sp500 open            float64
sp500 high            float64
sp500 low             float64
sp500 close           float64
sp500 volume          float64
sp500 high-low        float64
nasdaq open           float64
nasdaq high           float64
nasdaq low            float64
nasdaq close          float64
nasdaq volume         float64
nasdaq high-low       float64
us_rates_%            float64
CPI                   float64
usd_chf               float64
eur_usd               float64
GDP                   float64
silver open           float64
silver high           float64
silver low            float64
silver close          float64
silver volume         float64
silver high-low       float64
oil open              float64
oil high              float64
oil low               float64
oil close             float64
oil volume            float64
oil high-low          float64
platinum open         float64
platinum high         float64
platinum low          float64

# To datetime

In [7]:
if 'date' in df.columns:
    # 1. convert to datetime
    df['date'] = pd.to_datetime(df['date'])
    
    # 2. extract categories
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek # 0=Monday, 6=Sunday
    df['is_weekend'] = df['day_of_week'].isin([5, 6])
    # 3. dropping weekends since markets are closed and rows are empty
    df = df[df['is_weekend'] == False].copy()
    
    # 4. deleting cols that are not needed anymore
    df.drop(columns=['is_weekend', 'date'], inplace=True)
else:
    print("Column 'date' already processed and removed.")
print(df.head())


Column 'date' already processed and removed.
   sp500 open  sp500 high  sp500 low  sp500 close  sp500 volume  \
0      114.49      115.14     114.42       114.93   115646960.0   
1      114.73      114.84     113.20       113.64   212252769.0   
2         NaN         NaN        NaN          NaN           NaN   
3      113.62      115.13     113.59       115.06   138671890.0   
4      114.28      114.45     112.98       113.89   216330645.0   

   sp500 high-low  nasdaq open  nasdaq high  nasdaq low  nasdaq close  ...  \
0            0.72        46.26       46.520       46.22         46.39  ...   
1            1.64        46.46       46.550       45.65         45.85  ...   
2             NaN          NaN          NaN         NaN           NaN  ...   
3            1.54        45.96       46.640       45.95         46.59  ...   
4            1.47        46.27       46.604       45.43         45.92  ...   

   palladium volume  palladium high-low  gold open  gold high  gold low  \
0       

# Synthetic data

In [9]:
sentiment_map = {'Bearish': 0, 'Neutral': 1, 'Bullish': 2}
df['sentiment'] = np.random.choice(['Bearish', 'Neutral', 'Bullish'], len(df))
df['asset_type'] = np.random.choice(['Stock', 'Index', 'Commodity', 'Forex'], len(df))

# 1. Label Encoding (for Ordinal) if use such method, we create 1 col, where we put 0-2, model thinks that bullish is stronger than bearish
df['sentiment_encoded'] = df['sentiment'].map(sentiment_map)

# 2. One-Hot Encoding (for Nominal) if use such method, we create 4 col(type_Stock, type_Index...), where put only 1 or zero. If false than 0, if true - 1
df = pd.get_dummies(df, columns=['asset_type'], prefix='type')
print(df.columns)

Index(['sp500 open', 'sp500 high', 'sp500 low', 'sp500 close', 'sp500 volume',
       'sp500 high-low', 'nasdaq open', 'nasdaq high', 'nasdaq low',
       'nasdaq close', 'nasdaq volume', 'nasdaq high-low', 'us_rates_%', 'CPI',
       'usd_chf', 'eur_usd', 'GDP', 'silver open', 'silver high', 'silver low',
       'silver close', 'silver volume', 'silver high-low', 'oil open',
       'oil high', 'oil low', 'oil close', 'oil volume', 'oil high-low',
       'platinum open', 'platinum high', 'platinum low', 'platinum close',
       'platinum volume', 'platinum high-low', 'palladium open',
       'palladium high', 'palladium low', 'palladium close',
       'palladium volume', 'palladium high-low', 'gold open', 'gold high',
       'gold low', 'gold close', 'gold volume', 'year', 'month', 'day_of_week',
       'sentiment', 'sentiment_encoded', 'type_Commodity', 'type_Forex',
       'type_Index', 'type_Stock', 'type_Commodity', 'type_Forex',
       'type_Index', 'type_Stock'],
      dtype='obj

# Working with other string data

In [10]:
# creating 'dirty' col: " ID-101", "id-202 ", "$303"
df['raw_id'] = [f" ID-{np.random.randint(100, 999)} " if i%2==0 else f"${np.random.randint(100, 999)}" for i in range(len(df))]

# 1. .str accessor
df['raw_id'] = df['raw_id'].str.strip().str.upper().str.replace('$', '', regex=False)

# 2. Regex
df['numeric_id'] = df['raw_id'].str.extract(r'(\d+)').astype(int)
#The r before the quotes means "raw string" (to prevent Python from trying to process backslashes).
#\d is a special character that represents any digit (0–9).
#+ means "one or more." This means we're looking for a string of consecutive digits.
#( ) is a capturing group. Whatever's inside the brackets is what's returned by the extract method.
print(df["numeric_id"].head())


0    633
1    994
2    443
3    949
4    135
Name: numeric_id, dtype: int64


# Balanced encoding

In [11]:
# 100 unique IDs
df['trader_id'] = np.random.randint(1, 100, len(df)).astype(str)
print(df["trader_id"].head())
# Frequency Encoding
# count frequency and replace id with this value we cant use one hot, because it creates too many excessive columns
freq = df['trader_id'].value_counts()
df['trader_freq_encoded'] = df['trader_id'].map(freq)
print(df["trader_freq_encoded"].head())
df = pd.get_dummies(df, columns=['trader_id'], prefix='id')
print(df.columns)

0    83
1    56
2    80
3    18
4    33
Name: trader_id, dtype: object
0    49
1    28
2    57
3    43
4    35
Name: trader_freq_encoded, dtype: int64
Index(['sp500 open', 'sp500 high', 'sp500 low', 'sp500 close', 'sp500 volume',
       'sp500 high-low', 'nasdaq open', 'nasdaq high', 'nasdaq low',
       'nasdaq close',
       ...
       'id_90', 'id_91', 'id_92', 'id_93', 'id_94', 'id_95', 'id_96', 'id_97',
       'id_98', 'id_99'],
      dtype='object', length=161)


# Cleaning and final results

In [12]:
# clean temporary columns
cols_to_drop = ['sentiment', 'raw_id']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
cols_to_remove = [col for col in df.columns if col.startswith('id_') or col.startswith('type_')]
df.drop(columns=cols_to_remove, inplace=True)
print(f"Removed {len(cols_to_remove)} redundant ID columns.")

print("\n--- FINAL PROOF: Data Types ---")
print(df.dtypes.value_counts())
print("\n--- FINAL INFO ---")
df.info()

Removed 107 redundant ID columns.

--- FINAL PROOF: Data Types ---
float64    46
int32       3
int64       3
Name: count, dtype: int64

--- FINAL INFO ---
<class 'pandas.core.frame.DataFrame'>
Index: 3855 entries, 0 to 3903
Data columns (total 52 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sp500 open           3719 non-null   float64
 1   sp500 high           3719 non-null   float64
 2   sp500 low            3719 non-null   float64
 3   sp500 close          3719 non-null   float64
 4   sp500 volume         3719 non-null   float64
 5   sp500 high-low       3719 non-null   float64
 6   nasdaq open          3719 non-null   float64
 7   nasdaq high          3719 non-null   float64
 8   nasdaq low           3719 non-null   float64
 9   nasdaq close         3719 non-null   float64
 10  nasdaq volume        3719 non-null   float64
 11  nasdaq high-low      3719 non-null   float64
 12  us_rates_%           127 non-null    f

# In this work, I performed advanced feature engineering and data encoding on a financial dataset. The key accomplishments include:

## Temporal Analysis: 
Converted raw date strings into datetime objects and extracted seasonal features (Year, Month, Day of Week). I successfully filtered out weekend noise to align the data with stock market trading sessions.

## Categorical Transformation:

Applied Label Encoding to ordinal data (Market Sentiment) to preserve its inherent hierarchy.

Used One-Hot Encoding for nominal data (Asset Type) to prevent the model from assuming false mathematical relationships.

## String & Regex Operations: 
Cleaned "dirty" string identifiers using .str accessors and extracted purely numeric features via Regular Expressions (\d+).

## Balanced Encoding: 
Addressed high-cardinality data (Trader ID) using Frequency Encoding instead of One-Hot Encoding, which prevented a "dimensionality explosion" and kept the dataset efficient.